In [1]:
import os, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

V2 = r'D:\FYP\Eco-Twin\v2_clean'
OUT_DIR = os.path.join(V2, 'ecotwin_outputs', 'plots', 'slide_charts')
os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 12,
    'axes.edgecolor': '#444', 'axes.labelcolor': '#222',
    'text.color': '#222', 'xtick.color': '#333', 'ytick.color': '#333',
    'axes.titleweight': 'bold', 'axes.titlesize': 14,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
})

COLORS = {
    'Persistence': '#B0B7C0',
    'No-graph': '#F0A868',
    'Static-GCN': '#6C93C7',
    'WA-TGCN (dynamic)': '#2E8B57',
}
ORDER = ['Persistence', 'No-graph', 'Static-GCN', 'WA-TGCN (dynamic)']

bl = json.load(open(os.path.join(V2, 'ecotwin_outputs', 'results', 'baselines_metrics.json')))

# ---------------------------------------------------------------- C2: baseline comparison bar
fig, ax = plt.subplots(figsize=(7.5, 5))
vals = [bl['overall'][m]['rmse'] for m in ORDER]
bars = ax.bar(ORDER, vals, color=[COLORS[m] for m in ORDER], width=0.6, edgecolor='white', linewidth=1.5)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 1.2, f'{v:.1f}', ha='center', fontsize=13, fontweight='bold', color='#222')
ax.set_ylabel('Test RMSE (\u00b5g/m\u00b3)')
ax.set_title('Held-out test RMSE by model')
ax.set_ylim(0, 95)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(axis='x', labelsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'C2_baseline_comparison.png'), dpi=200, bbox_inches='tight')
plt.close()
print('saved C2')

# ---------------------------------------------------------------- D3: RMSE by horizon
fig, ax = plt.subplots(figsize=(8.5, 5))
hz = bl['by_horizon']
hours = list(range(1, 25))
for m in ORDER:
    y = [hz[m][f'+{h}h'] for h in hours]
    ax.plot(hours, y, marker='o', ms=4, lw=2.2, label=m, color=COLORS[m])
ax.set_xlabel('Forecast hour ahead')
ax.set_ylabel('RMSE (\u00b5g/m\u00b3)')
ax.set_title('Test RMSE by forecast horizon')
ax.set_xticks([1, 6, 12, 18, 24])
ax.legend(frameon=False, fontsize=10, loc='lower right')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'D3_rmse_by_horizon.png'), dpi=200, bbox_inches='tight')
plt.close()
print('saved D3')

# ---------------------------------------------------------------- D6: regime-stratified comparison (KEY)
fig, ax = plt.subplots(figsize=(9, 5.5))
regimes = ['Stable', 'Moderate', 'Volatile']
x = np.arange(len(regimes))
w = 0.2
for i, m in enumerate(ORDER):
    y = [bl['by_volatility_regime'][m][r] for r in regimes]
    bars = ax.bar(x + (i - 1.5) * w, y, w, label=m, color=COLORS[m], edgecolor='white', linewidth=1)
    for xi, yi in zip(x + (i - 1.5) * w, y):
        ax.text(xi, yi + 1.5, f'{yi:.0f}', ha='center', fontsize=8.5, color='#333')
ax.set_xticks(x); ax.set_xticklabels(regimes, fontsize=12)
ax.set_ylabel('RMSE (\u00b5g/m\u00b3)')
ax.set_title('RMSE by volatility regime \u2014 where the graph helps (and doesn\u2019t)')
ax.legend(frameon=False, fontsize=9.5, ncol=2, loc='upper left')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.25)
ax.text(0.99, 0.02, 'Volatile windows: ~34% of data, ~76% of total error',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9.5, style='italic', color='#555')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'D6_regime_comparison.png'), dpi=200, bbox_inches='tight')
plt.close()
print('saved D6')

# ---------------------------------------------------------------- B1: wind graph snapshot + B2 calm-vs-windy
df_st = pd.read_excel(os.path.join(V2, 'Delhi_list_v2.xlsx'))
df_st.columns = df_st.columns.str.strip().str.lower()
names = df_st['station'].str.strip().tolist()
lats = df_st['latitude'].values.astype(float)
lons = df_st['longitude'].values.astype(float)
N = len(names)

dyn = np.load(os.path.join(V2, 'dynamic_adj_matrix.npy'), mmap_mode='r')
edge_counts = (dyn[:3000] > 0).sum((1, 2))
windy_t = int(np.argmax(edge_counts))
calm_t = int(np.argmin(edge_counts))

_df_ts = pd.read_csv(os.path.join(V2, 'EcoTwin_Final_Dataset.csv'), usecols=['Timestamp'], parse_dates=['Timestamp'])
all_ts = np.array(sorted(_df_ts['Timestamp'].unique()))

def draw_graph(ax, A, title, top_k=60):
    ax.scatter(lons, lats, s=55, c='#2E8B57', edgecolor='white', linewidth=1, zorder=3)
    ii, jj = np.where(A > 0)
    ww = np.array(A[ii, jj])
    order = np.argsort(ww)[::-1][:top_k]
    wmax = ww.max() if len(ww) else 1
    for k in order:
        i, j = ii[k], jj[k]
        rel = ww[k] / wmax
        arr = FancyArrowPatch((lons[i], lats[i]), (lons[j], lats[j]),
                               arrowstyle='-|>', mutation_scale=8, shrinkA=4, shrinkB=4,
                               lw=0.6 + 2.0 * rel, alpha=0.2 + 0.55 * rel, color='#1f5fa8', zorder=2)
        ax.add_patch(arr)
    ax.set_title(title, fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)

fig, ax = plt.subplots(figsize=(7, 7))
draw_graph(ax, dyn[windy_t], f'Wind-aware graph A(t) \u2014 {pd.Timestamp(all_ts[windy_t]):%d %b %Y, %H:%M} (active hour)')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'B1_wind_graph_snapshot.png'), dpi=200, bbox_inches='tight')
plt.close()
print('saved B1', 'edges at windy_t:', int((dyn[windy_t]>0).sum()))

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
draw_graph(axes[0], dyn[calm_t], f'Calm \u2014 {pd.Timestamp(all_ts[calm_t]):%d %b %Y, %H:%M}  ({int((dyn[calm_t]>0).sum())} edges)')
draw_graph(axes[1], dyn[windy_t], f'Active \u2014 {pd.Timestamp(all_ts[windy_t]):%d %b %Y, %H:%M}  ({int((dyn[windy_t]>0).sum())} edges)')
fig.suptitle('The graph rewires hour to hour \u2014 static distance graphs cannot do this', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'B2_calm_vs_windy.png'), dpi=200, bbox_inches='tight')
plt.close()
print('saved B2')

# ---------------------------------------------------------------- D2: best / worst forecast window (from live dashboard data)
pb = json.load(open(os.path.join(V2, 'dashboard_export', 'playback.json')))
errs = []
for f in pb:
    a = np.array([s['actual_24h'] for s in f['stations']], float)
    p = np.array([s['predicted_24h'] for s in f['stations']], float)
    errs.append(float(np.sqrt(((a - p) ** 2).mean())))
errs = np.array(errs)
best_i, worst_i = int(errs.argmin()), int(errs.argmax())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, idx, lbl, col in [(axes[0], best_i, 'Best window', '#2E8B57'), (axes[1], worst_i, 'Hardest window', '#C0392B')]:
    frame = pb[idx]
    sdata = frame['stations'][0]  # station 0 for a clean single-line read; swap if needed
    hrs = list(range(1, 25))
    ax.plot(hrs, sdata['actual_24h'], marker='o', ms=4, lw=2, color='#333', label='Actual')
    ax.plot(hrs, sdata['predicted_24h'], marker='s', ms=4, lw=2, ls='--', color=col, label='WA-TGCN forecast')
    ax.set_title(f"{lbl} \u2014 {frame['timestamp']}  (RMSE {errs[idx]:.0f})", fontsize=11.5)
    ax.set_xlabel('Forecast hour ahead'); ax.set_ylabel('PM2.5 (\u00b5g/m\u00b3)')
    ax.legend(frameon=False, fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'D2_best_worst_window.png'), dpi=200, bbox_inches='tight')
plt.close()
print('saved D2  best', pb[best_i]['timestamp'], 'worst', pb[worst_i]['timestamp'])

print('\nALL CHARTS SAVED TO', OUT_DIR)

saved C2
saved D3
saved D6
saved B1 edges at windy_t: 306
saved B2
saved D2  best 2026-03-05 01:00 worst 2026-01-17 09:00

ALL CHARTS SAVED TO D:\FYP\Eco-Twin\v2_clean\ecotwin_outputs\plots\slide_charts


In [5]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

V2 = r'D:\FYP\Eco-Twin\v2_clean'
OUT_DIR = os.path.join(V2, 'ecotwin_outputs', 'plots', 'slide_charts')
os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 12,
    'axes.edgecolor': '#444', 'axes.labelcolor': '#222',
    'text.color': '#222', 'xtick.color': '#333', 'ytick.color': '#333',
    'axes.titleweight': 'bold', 'axes.titlesize': 14,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
})

In [6]:
hist = json.load(open(os.path.join(V2, 'ecotwin_outputs', 'results', 'training_history.json')))
epochs = list(range(1, len(hist['train_loss']) + 1))

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(epochs, hist['train_loss'], marker='o', ms=5, lw=2.2, color='#2E8B57', label='Train loss')
ax.plot(epochs, hist['val_loss'], marker='s', ms=5, lw=2.2, color='#C0392B', label='Val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Volatility-weighted Huber loss')
ax.set_title('Training convergence \u2014 WA-TGCN v2')
ax.set_xticks(epochs)
ax.legend(frameon=False, fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'C1_loss_curve.png'), dpi=200, bbox_inches='tight')
plt.show()
print('saved C1, epochs:', len(epochs))

saved C1, epochs: 12


C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_20576\262397739.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
df_st = pd.read_excel(os.path.join(V2, 'Delhi_list_v2.xlsx'))
df_st.columns = df_st.columns.str.strip().str.lower()
lats = df_st['latitude'].values.astype(float)
lons = df_st['longitude'].values.astype(float)

dyn = np.load(os.path.join(V2, 'dynamic_adj_matrix.npy'), mmap_mode='r')
edge_counts = (dyn[:3000] > 0).sum((1, 2))
windy_t = int(np.argmax(edge_counts))
calm_t = int(np.argmin(edge_counts))

_df_ts = pd.read_csv(os.path.join(V2, 'EcoTwin_Final_Dataset.csv'), usecols=['Timestamp'], parse_dates=['Timestamp'])
all_ts = np.array(sorted(_df_ts['Timestamp'].unique()))

def draw_graph(ax, A, title, top_k=60):
    ax.scatter(lons, lats, s=55, c='#2E8B57', edgecolor='white', linewidth=1, zorder=3)
    ii, jj = np.where(A > 0)
    ww = np.array(A[ii, jj])
    order = np.argsort(ww)[::-1][:top_k]
    wmax = ww.max() if len(ww) else 1
    for k in order:
        i, j = ii[k], jj[k]
        rel = ww[k] / wmax
        arr = FancyArrowPatch((lons[i], lats[i]), (lons[j], lats[j]),
                               arrowstyle='-|>', mutation_scale=8, shrinkA=4, shrinkB=4,
                               lw=0.6 + 2.0 * rel, alpha=0.2 + 0.55 * rel, color='#1f5fa8', zorder=2)
        ax.add_patch(arr)
    ax.set_title(title, fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
draw_graph(axes[0], dyn[calm_t], f'Calm \u2014 {pd.Timestamp(all_ts[calm_t]):%d %b %Y, %H:%M}  ({int((dyn[calm_t]>0).sum())} edges)')
draw_graph(axes[1], dyn[windy_t], f'Active \u2014 {pd.Timestamp(all_ts[windy_t]):%d %b %Y, %H:%M}  ({int((dyn[windy_t]>0).sum())} edges)')
fig.suptitle('The graph rewires hour to hour \u2014 static distance graphs cannot do this', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'B2_calm_vs_windy.png'), dpi=200, bbox_inches='tight')
plt.show()
print('saved B2')

saved B2


C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_20576\116650222.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
pb = json.load(open(os.path.join(V2, 'dashboard_export', 'playback.json')))
errs = []
for f in pb:
    a = np.array([s['actual_24h'] for s in f['stations']], float)
    p = np.array([s['predicted_24h'] for s in f['stations']], float)
    errs.append(float(np.sqrt(((a - p) ** 2).mean())))
errs = np.array(errs)
best_i, worst_i = int(errs.argmin()), int(errs.argmax())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, idx, lbl, col in [(axes[0], best_i, 'Best window', '#2E8B57'), (axes[1], worst_i, 'Hardest window', '#C0392B')]:
    frame = pb[idx]
    sdata = frame['stations'][0]
    hrs = list(range(1, 25))
    ax.plot(hrs, sdata['actual_24h'], marker='o', ms=4, lw=2, color='#333', label='Actual')
    ax.plot(hrs, sdata['predicted_24h'], marker='s', ms=4, lw=2, ls='--', color=col, label='WA-TGCN forecast')
    ax.set_title(f"{lbl} \u2014 {frame['timestamp']}  (RMSE {errs[idx]:.0f})", fontsize=11.5)
    ax.set_xlabel('Forecast hour ahead'); ax.set_ylabel('PM2.5 (\u00b5g/m\u00b3)')
    ax.legend(frameon=False, fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'D2_best_worst_window.png'), dpi=200, bbox_inches='tight')
plt.show()
print('saved D2, best', pb[best_i]['timestamp'], 'worst', pb[worst_i]['timestamp'])

saved D2, best 2026-03-05 01:00 worst 2026-01-17 09:00


C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_20576\188532391.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
df_st = pd.read_excel(os.path.join(V2, 'Delhi_list_v2.xlsx'))
df_st.columns = df_st.columns.str.strip().str.lower()
names = df_st['station'].str.strip().tolist()
lats = df_st['latitude'].values.astype(float)
lons = df_st['longitude'].values.astype(float)

dyn = np.load(os.path.join(V2, 'dynamic_adj_matrix.npy'), mmap_mode='r')
edge_counts = (dyn[:3000] > 0).sum((1, 2))
windy_t = int(np.argmax(edge_counts))

_df_ts = pd.read_csv(os.path.join(V2, 'EcoTwin_Final_Dataset.csv'), usecols=['Timestamp'], parse_dates=['Timestamp'])
all_ts = np.array(sorted(_df_ts['Timestamp'].unique()))
target_ts = pd.Timestamp(all_ts[windy_t])

wind_df = pd.read_csv(os.path.join(V2, 'EcoTwin_Final_Dataset.csv'),
                       usecols=['Timestamp', 'Station', 'u_wind', 'v_wind'],
                       parse_dates=['Timestamp'])
wind_df['Station'] = wind_df['Station'].str.strip()
snap = wind_df[wind_df['Timestamp'] == target_ts].set_index('Station')
u = np.array([snap.loc[n, 'u_wind'] for n in names])
v = np.array([snap.loc[n, 'v_wind'] for n in names])

A = dyn[windy_t]
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(lons, lats, s=55, c='#222', zorder=4)

ii, jj = np.where(A > 0)
ww = np.array(A[ii, jj])
order = np.argsort(ww)[::-1][:60]
wmax = ww.max() if len(ww) else 1
for k in order:
    i, j = ii[k], jj[k]
    rel = ww[k] / wmax
    arr = FancyArrowPatch((lons[i], lats[i]), (lons[j], lats[j]),
                           arrowstyle='-|>', mutation_scale=9, shrinkA=4, shrinkB=4,
                           lw=0.6 + 2.0 * rel, alpha=0.25 + 0.55 * rel, color='#1f5fa8', zorder=2,
                           label='Graph edge (downwind, weighted)' if k == order[0] else None)
    ax.add_patch(arr)

scale = 0.012
ax.quiver(lons, lats, u, v, color='#C0392B', width=0.004, scale=1/scale, scale_units='xy',
          zorder=3, label='Actual wind vector (u,v)')

ax.set_title(f'Graph edges follow real wind \u2014 {target_ts:%d %b %Y, %H:%M}', fontsize=13)
ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values(): s.set_visible(False)
ax.legend(loc='upper left', frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'B3_wind_vectors_vs_graph.png'), dpi=200, bbox_inches='tight')
plt.show()
print('saved B3')

saved B3


C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_20576\1251599170.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
